# Hafta 4 – 01: Doğru, güvenli ve bankaya yakışır yanıt

> Hafta 4 sabah notebook'u (Lab 1). Asistan doğru bilgiyi bulmayı biliyor (Hafta 3). Bu hafta ona bu bilgiyi
> **doğru, güvenli ve bankaya yakışır** biçimde söylemeyi öğretiyoruz ve iyileştiğini **ölçerek** kanıtlıyoruz.

Notebook boyunca **tek bir talep** kullanıyoruz: *"Kartımdan aidat iki kez kesildi"* (müşteri **M0007**).
Her bölüm tek bir soruya cevap verir ve yanıtın **önceki hali ile yeni halini yan yana** gösterir.

| Bölüm | Soru | Projede nerede |
|---|---|---|
| 1 | İyi bir talimat nasıl yazılır? | `prompts/taslak/v2.md`, `src/asistan/sablonlar.py` |
| 2 | Müşteriye ne gider? | `prompts/banka_dili.md`, `src/asistan/maskeleme.py` |
| 3 | Gönderilmeden önce kim kontrol ediyor? | `src/asistan/self_check.py`, `prompts/self_check/v1.md` |
| 4 | Gerçekten iyileşti mi? | `eval/ab_testi.py`, `eval/hakem.py` |

Her kod hücresinin üstünde iki cümle var: **Ne yapıyoruz?** ve **Neye bakacağız?** Hücreyi çalıştırmadan önce okuyun.

**Ne yapıyoruz?** Notebook'u projenin koduna bağlıyor, talebi ve modeli seçiyoruz.
**Neye bakacağız?** Son satırda model ve talep yazmalı; hata varsa önce `python kontrol.py` çalıştırın.

In [1]:
import os, sys, textwrap
from itertools import zip_longest
from pathlib import Path

PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))   # asistan paketi
sys.path.insert(0, str(PROJE_KOKU / "eval"))  # hakem.py (Bölüm 4)

MODEL = "bulut"     # "yerel" de çalışır; küçük modelde sonuçlar daha değişken olur
TALEP = "Kartımdan aidat iki kez kesildi"
MUSTERI = "M0007"


def yan_yana(sol_baslik, sol, sag_baslik, sag, genislik=52):
    """İki metni iki sütun halinde yazar: solda önceki hal, sağda yeni hal."""
    sol_satir = [s for p in sol.split("\n") for s in (textwrap.wrap(p, genislik) or [""])]
    sag_satir = [s for p in sag.split("\n") for s in (textwrap.wrap(p, genislik) or [""])]
    print(f"{sol_baslik:<{genislik}} │ {sag_baslik}\n{'─' * genislik}─┼─{'─' * genislik}")
    for a, b in zip_longest(sol_satir, sag_satir, fillvalue=""):
        print(f"{a:<{genislik}} │ {b}")


print(f"Model: {MODEL} | Talep: {TALEP!r} ({MUSTERI})")

Model: bulut | Talep: 'Kartımdan aidat iki kez kesildi' (M0007)


## 1. İyi bir talimat nasıl yazılır?

Hafta 3'te istemler (prompt) Python kodunun içinde uzun metinlerdi. Bu hafta hepsi `prompts/` klasöründe,
**sürümlü** dosyalar: `prompts/taslak/v1.md` (Hafta 3'ün metni, birebir) ve `prompts/taslak/v2.md` (yeni).
v2 dört bölümden oluşur: **rol** (kimsin), **görev** (ne yapacaksın), **bağlam** (elinde ne var), **kısıtlar** (ne yapmayacaksın).

**Ne yapıyoruz?** `taslak/v2.md` şablonunun iskeletine bakıyoruz: bölüm başlıkları ve içindeki yer tutucular.
**Neye bakacağız?** Dört bölümü ve `{kurallar}`, `{banka_dili}`, `{prosedur_parcalari}` gibi süslü parantezli yer tutucuları.

In [2]:
from asistan.sablonlar import SABLON_KLASORU, YER_TUTUCU, dosya_oku

sablon = dosya_oku(SABLON_KLASORU / "taslak" / "v2.md")
for satir in sablon.splitlines():
    if satir.startswith(("## Rol", "## Görev", "## Bağlam", "## Kısıtlar", "===")):
        print(satir)
print("\nYer tutucular:", ", ".join(dict.fromkeys(YER_TUTUCU.findall(sablon))))

## Rol
## Görev
## Bağlam
## Kısıtlar
=== KULLANICI MESAJI ===

Yer tutucular: kurallar, banka_dili, talep, kategori, ikinci_konu, musteri_ozeti, prosedur_parcalari, harici_veri, ajan_notu, duzeltme


**Ne yapıyoruz?** Asistanı talebimizle bir kez çalıştırıyor ve taslak adımına giden verileri (yer tutucuların değerlerini) alıyoruz.
**Neye bakacağız?** Her yer tutucunun neyle dolduğuna: müşteri verisi, prosedür parçaları, harici veri. Kart numarası yerine `[KART_1]` görmelisiniz.

In [3]:
from asistan import cevapla
from asistan.ajan import GRAF, taslak_baglami

sonuc = cevapla(TALEP, MUSTERI, MODEL)                     # asistanın tüm akışı (bir kez)
durum = GRAF.get_state({"configurable": {"thread_id": sonuc["oturum_id"]}}).values
degerler = taslak_baglami(durum)                            # taslak şablonunun yer tutucu değerleri

for ad, deger in degerler.items():
    print(f"{'{' + ad + '}':<22} {deger.replace(chr(10), ' ')[:75]}")

{talep}                Kartımdan aidat iki kez kesildi
{kategori}             kart
{ikinci_konu}          yok
{musteri_ozeti}        <arac_verisi arac="kart_islemleri"> {"islem_id": "I00056", "musteri_no": "M
{harici_veri}          (harici veri yok)
{prosedur_parcalari}   <prosedur kaynak="kart/kart_aidati.md#Mükerrer aidat"> - Aynı kart için ayn
{ajan_notu}            Bulgular:  - **Mükerrer aidat çekimi var.** 3 Eylül 2026 tarihinde, 0007 il


**Ne yapıyoruz?** Şablonu bu değerlerle dolduruyoruz (`sablonlar.mesajlar`). Üst kısım **sistem mesajı** (talimat), alt kısım **kullanıcı mesajı** (veri) olur.
**Neye bakacağız?** Talimat ile verinin ayrı mesajlarda olduğuna ve prosedür parçalarının `<prosedur kaynak="...">` etiketiyle geldiğine.

In [4]:
from asistan.sablonlar import mesajlar, surum_sec

SURUM = surum_sec("taslak", MODEL)                          # hangi sürüm? (SURUMLER ya da .env)
istem = mesajlar("taslak", MODEL, **degerler, duzeltme="")
(_, sistem), (_, kullanici) = istem
print(f"Taslak şablonu: {SURUM}")
print(f"Sistem mesajı: {len(sistem.splitlines())} satır (rol, görev + kurallar.md, bağlam, kısıtlar + banka_dili.md)")
print(f"Kullanıcı mesajı: {len(kullanici.splitlines())} satır. Prosedür bölümünden bir kesit:\n")
bas = kullanici.index("## Prosedür parçaları")
print(kullanici[bas:bas + 420], "...")

Taslak şablonu: v2
Sistem mesajı: 94 satır (rol, görev + kurallar.md, bağlam, kısıtlar + banka_dili.md)
Kullanıcı mesajı: 43 satır. Prosedür bölümünden bir kesit:

## Prosedür parçaları
<prosedur kaynak="kart/kart_aidati.md#Mükerrer aidat">
- Aynı kart için aynı yıl içinde birden fazla aidat tahsil edilmişse fazla olan tahsilat sistem hatasıdır
  ve iade edilir. Mükerrer olduğu kart işlemlerinden (aynı gün, aynı tutar, işlem tipi aidat) doğrulanır.
- İade edilecek olan, sonra gerçekleşen (ikinci) tahsilattır.
</prosedur>

## Harici veri
(harici veri yok)

## Ajanın notu
Bulgula ...


**Ne yapıyoruz?** Aynı veriyi iki farklı talimatla modele veriyoruz: tek cümlelik **zayıf** bir istem ve dört bölümlü **taslak v2**.
**Neye bakacağız?** Zayıf istemde söz verme ("iade edilecektir"), kaynak eksikliği ve uzunluk; v2'de karar, kaynak etiketi ve ölçülü dil.

In [5]:
from asistan.ajan import Taslak
from asistan.ayarlar import model_getir

ZAYIF = "Sen bir banka asistanısın. Müşteriye yanıt yaz."
zayif_yanit = model_getir(MODEL).invoke([("system", ZAYIF), ("human", kullanici)]).text
v2 = model_getir(MODEL).with_structured_output(Taslak, method="json_schema" if MODEL != "yerel" else "function_calling").invoke(istem)

yan_yana("ÖNCE: zayıf istem", zayif_yanit, f"SONRA: taslak {SURUM} (karar: {v2.karar})", v2.yanit)

ÖNCE: zayıf istem                                    │ SONRA: taslak v2 (karar: yanitla)
─────────────────────────────────────────────────────┼─────────────────────────────────────────────────────
Merhaba,                                             │ Bu durumun can sıkıcı olduğunu anlıyoruz. [KART_1]
                                                     │ üzerinden 3 Eylül 2026'da aynı tutarda iki aidat
Kartınızın hareketlerini inceledim. 0007 ile biten   │ kesildiğini gördük. Aynı yıl içinde fazla alınan
Örnek Classic Kredi Kartı'ndan 3 Eylül 2026          │ aidat iade edilir ve iade edilecek olan sonraki
tarihinde "ORNEK BANK KART YILLIK AIDATI" adıyla     │ tahsilattır [kaynak: kart/kart_aidati.md#Mükerrer
aynı gün ve aynı tutarda (750,00 TL) iki ayrı        │ aidat]. İkinci tahsilat için iade talebiniz onaya
tahsilat yapıldığını gördüm. Yıllık aidat bir kez    │ iletildi.
alınması gereken bir ücret olduğundan, ikinci        │ 
tahsilat mükerrerdir.                                │ 

## 2. Müşteriye ne gider?

İki soru: (1) **Modele** ne gidiyor? Kart numarası, IBAN, TCKN buluta gitmemeli. (2) **Müşteriye** ne gidiyor?
Tam numara hiç yazılmamalı, kaynak okunur olmalı, dil `prompts/banka_dili.md`'ye uymalı.
Bunun için `maskeleme.py` her kişisel veriye bir **takma ad** verir (`[KART_1]`) ve eşlemeyi sadece bellekte tutar.

**Ne yapıyoruz?** `musteri_bilgisi` aracının servisten aldığı gerçek veriyi, modele gitmeden önceki maskeli haliyle karşılaştırıyoruz.
**Neye bakacağız?** Solda gerçek ad, TCKN, IBAN, kart no; sağda `[AD_1]`, `[TCKN_1]`, `[IBAN_1]`, `[KART_1]`. Limit ve durum gibi alanlar değişmez.

In [6]:
import json
from asistan.araclar import api_cagir
from asistan.maskeleme import veriyi_maskele, yeni_tablo

ham = api_cagir("GET", f"/musteri_bilgisi/{MUSTERI}")
alanlar = {k: ham[k] for k in ("ad_soyad", "tckn", "telefon", "iban")} | {"kart": ham["kartlar"][0]}
ozet = lambda v: json.dumps(v, ensure_ascii=False, indent=1)

yan_yana("ÖNCE: servisten gelen (gerçek)", ozet(alanlar), "SONRA: modele giden (takma ad)", ozet(veriyi_maskele(alanlar, yeni_tablo())))

ÖNCE: servisten gelen (gerçek)                       │ SONRA: modele giden (takma ad)
─────────────────────────────────────────────────────┼─────────────────────────────────────────────────────
{                                                    │ {
 "ad_soyad": "Selin Prova",                          │  "ad_soyad": "[AD_1]",
 "tckn": "99999000007",                              │  "tckn": "[TCKN_1]",
 "telefon": "0500 000 0007",                         │  "telefon": "[TEL_1]",
 "iban": "TR00 9999 9000 0000 0000 0000 07",         │  "iban": "[IBAN_1]",
 "kart": {                                           │  "kart": {
  "kart_no": "9999 0000 0000 0007",                  │   "kart_no": "[KART_1]",
  "kart_adi": "Örnek Classic Kredi Kartı",           │   "kart_adi": "Örnek Classic Kredi Kartı",
  "kart_tipi": "kredi",                              │   "kart_tipi": "kredi",
  "limit": 25000,                                    │   "limit": 25000,
  "son_kullanma": "2029-06",                 

**Ne yapıyoruz?** Bölüm 1'deki çalıştırmanın taslağını iki halde gösteriyoruz: modelin yazdığı (takma adlı) ve müşteriye giden.
**Neye bakacağız?** `[KART_1]` → "0007 ile biten kartınız"; `[kaynak: kart/kart_aidati.md#...]` → yanıtın sonunda "Kaynak: Kart Aidatı – ..." satırı.

In [7]:
yan_yana("ÖNCE: modelin yazdığı taslak", durum["taslak"]["yanit"], "SONRA: müşteriye giden yanıt", sonuc["yanit"])

ÖNCE: modelin yazdığı taslak                         │ SONRA: müşteriye giden yanıt
─────────────────────────────────────────────────────┼─────────────────────────────────────────────────────
Bize yazdığınız için teşekkür ederiz. [KART_1]       │ Bize yazdığınız için teşekkür ederiz. 0007 ile biten
üzerinden aynı gün iki kez aidat kesildiğini gördük; │ kartınız üzerinden aynı gün iki kez aidat
fazla olan tahsilat sistem hatası sayılıyor ve       │ kesildiğini gördük; fazla olan tahsilat sistem
sonraki (ikinci) tahsilat iade edilir [kaynak:       │ hatası sayılıyor ve sonraki (ikinci) tahsilat iade
kart/kart_aidati.md#Mükerrer aidat]. İade talebiniz  │ edilir. İade talebiniz onaya iletildi. Bu durumun
onaya iletildi. Bu durumun can sıkıcı olduğunu       │ can sıkıcı olduğunu anlıyoruz.
anlıyoruz.                                           │ 
                                                     │ Kaynak: Kart Aidatı – Mükerrer aidat


**Ne yapıyoruz?** Bu çalıştırmanın izini (`logs/izler/`) açıp her model çağrısında modele maskesiz kişisel veri gidip gitmediğine bakıyoruz.
**Neye bakacağız?** `maskesiz veri` sütununda her satır "yok" olmalı. Bu, buluta kart numarası gitmediğinin kanıtıdır.

In [8]:
iz = [json.loads(s) for d in sorted((PROJE_KOKU / "logs" / "izler").glob("*.jsonl"))[-1:]
      for s in open(d, encoding="utf-8") if sonuc["oturum_id"] in s][-1]
print("İzdeki talep    :", iz["talep"])
print("Şablon sürümleri:", iz["sablonlar"], "\n")
print(f"{'düğüm':<14}{'girdi token':>12}   maskesiz veri")
for o in (o for o in iz["olaylar"] if o["tur"] == "model"):
    print(f"{o['dugum']:<14}{o['girdi_token'] or 0:>12}   {', '.join(o['maskesiz_veri']) or 'yok'}")

İzdeki talep    : Kartımdan aidat iki kez kesildi
Şablon sürümleri: {'siniflandirma': 'v1', 'ajan': 'v1', 'taslak': 'v2', 'self_check': 'v1'} 

düğüm          girdi token   maskesiz veri
siniflandir           2091   yok
ajan                  2338   yok
ajan                  3961   yok
ajan                  4302   yok
taslak                6051   yok
self_check            4723   yok


## 3. Gönderilmeden önce kim kontrol ediyor?

Taslak yazıldıktan sonra **self_check** düğümü çalışır (`self_check.py`). İki katmanı var:
1. **Kod kontrolleri** (her zaman aynı sonuç, ücretsiz): kaynak gerçekten getirilmiş mi, tam numara kalmış mı, yasak ifade var mı, iç doküman alıntılanmış mı?
2. **Model eleştirisi:** ikinci bir model çağrısı kontrol listesini doldurur: kaynakla tutarlı mı, soruyu cevaplıyor mu, banka diline uygun mu, uydurma var mı, iade/ödeme/taahhüt içeriyor mu?

Geçmezse taslak gerekçesiyle **bir kez** yeniden yazılır; yine geçmezse talep temsilciye aktarılır.

**Ne yapıyoruz?** Kötü bir taslak uyduruyoruz (modelin yazabileceği türden) ve önce **kod kontrollerinden** geçiriyoruz.
**Neye bakacağız?** Kod dört sorunu yakalamalı: verilmemiş kaynak, tam kart numarası ve iki yasak ifade.

In [9]:
from asistan.ajan import kaynaklari_dogrula
from asistan.rag.arama import baglam_olustur
from asistan.self_check import deterministik_kontroller

HATALI = ("Kartınızdan kesilen aidat kesinlikle iade edildi, 9999 0000 0000 0007 numaralı kartınıza yansıdı "
          "[kaynak: kart/aidat_kampanyasi.md#İade].")
baglam = baglam_olustur(durum["getirilen_parcalar"])
temiz, gecerli, uydurma = kaynaklari_dogrula(HATALI, [], baglam)   # verilmemiş kaynak silinir, listesi döner
for sorun in deterministik_kontroller(temiz, uydurma):
    print("✗", sorun)

✗ sana verilmeyen kaynak gösterildi: kart/aidat_kampanyasi.md#İade
✗ yanıtta tam kişisel veri var: KART
✗ yasak ifade: kesinlikle, iade edildi


**Ne yapıyoruz?** Aynı kötü taslağı **model eleştirisine** veriyoruz (takma adlı haliyle; modele numara gitmez).
**Neye bakacağız?** `uydurma_var` ve `iade_odeme_taahhut` alanlarına: "iade edildi" yapılmamış bir işlem ve bir söz.

In [10]:
from asistan.maskeleme import maske_oturumu, metni_maskele
from asistan.self_check import elestir

with maske_oturumu(sonuc["oturum_id"]):
    hatali_maskeli = metni_maskele(HATALI)
elestiri = elestir(MODEL, **{k: v for k, v in degerler.items() if k != "ajan_notu"},
                   taslak=hatali_maskeli, iade_onerisi="yok")
for alan, deger in elestiri.model_dump().items():
    print(f"{alan:<20} {deger}")

gerekce              Yasak 'kesinlikle' ve 'iade edildi' ifadelerini çıkarın; iade yapılmadığı halde yapılmış gibi gösteriliyor (uydurma), kaynak etiketi prosedürde olmayan bir dosyaya işaret ediyor, ayrıca yanıt iki kez kesilen 750 TL aidatı (I00287/I00288, 3 Eylül) ve ikinci tahsilatın iade sürecini doğru aktarmıyor ve bankanın adına iade sözü veriyor.
kaynakla_tutarli     False
soruyu_cevapliyor    False
banka_diline_uygun   False
uydurma_var          True
iade_odeme_taahhut   evet


**Ne yapıyoruz?** Akışın yaptığını elle yapıyoruz: kontrolün gerekçesini taslak şablonuna **düzeltme isteği** olarak ekleyip yeniden yazdırıyoruz.
**Neye bakacağız?** Sağdaki yanıtta "kesinlikle", "iade edildi" ve uydurma kaynak olmamalı; iade "onaya iletildi" diye anlatılmalı.

In [11]:
sorunlar = deterministik_kontroller(temiz, uydurma) + [f"eleştiri: {elestiri.gerekce}"]
duzeltme = ("\n\n## Düzeltme isteği\nÖnceki taslağın kontrolden geçmedi: " + "; ".join(sorunlar)
            + "\nBu sorunları gidererek taslağı yeniden yaz.")
yeni = model_getir(MODEL).with_structured_output(Taslak, method="json_schema" if MODEL != "yerel" else "function_calling") \
    .invoke(mesajlar("taslak", MODEL, **degerler, duzeltme=duzeltme))

yan_yana("ÖNCE: kontrolden geçemeyen taslak", HATALI, "SONRA: yeniden yazılan taslak", yeni.yanit)

ÖNCE: kontrolden geçemeyen taslak                    │ SONRA: yeniden yazılan taslak
─────────────────────────────────────────────────────┼─────────────────────────────────────────────────────
Kartınızdan kesilen aidat kesinlikle iade edildi,    │ Bize yazdığınız için teşekkür ederiz. Kartınızdan 3
9999 0000 0000 0007 numaralı kartınıza yansıdı       │ Eylül 2026'da aynı gün iki kez aidat kesildiğini
[kaynak: kart/aidat_kampanyasi.md#İade].             │ görüyoruz; bu durumun can sıkıcı olduğunu anlıyoruz.
                                                     │ Fazla olan ikinci tahsilat için iade talebiniz onaya
                                                     │ iletildi [kaynak: kart/kart_aidati.md#Mükerrer
                                                     │ aidat]. Sonuç hakkında bilgi vermemiz için başka bir
                                                     │ işlem yapmanıza gerek yok.


**Ne yapıyoruz?** Yeniden yazılan taslağı tekrar kontrol ediyor ve **risk kontrolünün** ne karar vereceğine bakıyoruz.
**Neye bakacağız?** Kontrol geçmeli. İade önerisi olduğu için taahhüt "evet" ve karar **insan onayı** olmalı; para işlemi insansız gitmez.

In [12]:
from asistan.ajan import risk_nedenleri_bul

kontrol = deterministik_kontroller(yeni.yanit, [])
elestiri2 = elestir(MODEL, **{k: v for k, v in degerler.items() if k != "ajan_notu"}, taslak=yeni.yanit,
                    iade_onerisi=f"{yeni.iade_islem_id}, {yeni.iade_tutari} TL" if yeni.iade_islem_id else "yok")
print("Kod kontrolleri :", "geçti" if not kontrol else kontrol)
print("Model eleştirisi:", "geçti" if elestiri2.kaynakla_tutarli and not elestiri2.uydurma_var else elestiri2.gerekce)
print("Taahhüt         :", elestiri2.iade_odeme_taahhut, f"(iade önerisi: {yeni.iade_islem_id or 'yok'})")
nedenler = risk_nedenleri_bul(yeni.yanit, TALEP, bool(yeni.iade_islem_id), elestiri2.iade_odeme_taahhut)
print("Risk kontrolü   :", ", ".join(nedenler) or "risk yok", "->", "insan_onayi" if nedenler else "yanitla")

Kod kontrolleri : geçti
Model eleştirisi: Son cümle müşteriye sonuç hakkında bilgi verileceği sözünü veriyor, ancak bu prosedürde yazmıyor; bu ifadeyi çıkarın ya da prosedürde yazan süre ve sonuç bilgisiyle değiştirin.
Taahhüt         : evet (iade önerisi: I00288)
Risk kontrolü   : iade, taahhut -> insan_onayi


## 4. Gerçekten iyileşti mi?

"Daha iyi göründü" yetmez; **ölçeriz**. İki araç:
- **A/B testi** (`eval/ab_testi.py`): aynı talepler, iki şablon sürümü, aynı metrikler. Yerel modelde sonuçlar koşudan koşuya oynar; bu yüzden birkaç kez çalıştırırız.
- **Hakem** (`eval/hakem.py`): güçlü bir model yanıtları 1-5 arası puanlar (doğruluk, eksiksizlik, banka dili, empati) ve her puanın gerekçesini yazar.

**Ne yapıyoruz?** Küçük bir A/B testi: aynı talebi taslak **v1** (Hafta 3 metni) ve **v2** ile ikişer kez çalıştırıyoruz.
**Neye bakacağız?** Davranışlar aynı mı, yanıtlar nasıl farklı? Aynı sürümün iki koşusu bile birebir aynı değildir.

In [13]:
ab = {}
for surum in ("v1", "v2"):
    os.environ["SABLON_TASLAK"] = surum          # sürüm .env/ortamdan seçilir; kod değişmez
    ab[surum] = [cevapla(TALEP, MUSTERI, MODEL) for _ in range(2)]
os.environ.pop("SABLON_TASLAK")

for surum, kosular in ab.items():
    print(f"taslak {surum}: davranışlar = {[k['davranis'] for k in kosular]}")
print()
yan_yana("ÖNCE: taslak v1 (Hafta 3 metni)", ab["v1"][0]["yanit"], "SONRA: taslak v2", ab["v2"][0]["yanit"])

taslak v1: davranışlar = ['insan_onayi', 'insan_onayi']
taslak v2: davranışlar = ['insan_onayi', 'insan_onayi']

ÖNCE: taslak v1 (Hafta 3 metni)                      │ SONRA: taslak v2
─────────────────────────────────────────────────────┼─────────────────────────────────────────────────────
0007 ile biten kartınızda 03.09.2026 tarihinde       │ Bu durumun can sıkıcı olduğunu anlıyoruz. 0007 ile
yıllık aidatın iki kez tahsil edildiğini tespit      │ biten kartınız üzerinden 3 Eylül 2026'da aidat iki
ettik; aynı yıl içinde fazla alınan aidat sistem     │ kez kesilmiş; fazla olan ikinci tahsilat için iade
hatası sayılmakta ve iade edilmektedir. İkinci       │ talebiniz onaya iletildi.
tahsilata ait iade talebiniz temsilci onayına        │ 
iletilmiştir. Onaylanan iade 3 iş günü içinde kart   │ Kaynak: Kart Aidatı – Mükerrer aidat
hesabınıza yansır.                                   │ 
                                                     │ 
Kaynak: Kart Aidatı – Mükerrer aidat            

**Ne yapıyoruz?** İki sürümün ilk yanıtını **hakeme** puanlatıyoruz (güçlü model, `CLAUDE_MODEL_GUCLU`).
**Neye bakacağız?** Hangi ölçütte fark var ve hakemin gerekçesi bu farkı açıklıyor mu? Hakem de yanılabilir: gerekçeyi okuyun.

In [14]:
from hakem import OLCUTLER, puanla

puanlar = {surum: puanla({"id": surum, "talep": TALEP, "tahmin_davranis": k[0]["davranis"], "yanit": k[0]["yanit"],
                          "kaynaklar": ";".join(k[0]["kaynaklar"]),
                          "not": "M0007'de 2026-09-03'te iki aidat (I00287, I00288); iade önerisi I00288 olmalı"})
           for surum, k in ab.items()}
print(f"{'ölçüt':<13}{'v1':>4}{'v2':>4}   v2 gerekçesi")
for o in OLCUTLER:
    print(f"{o:<13}{getattr(puanlar['v1'], o):>4}{getattr(puanlar['v2'], o):>4}   {getattr(puanlar['v2'], o + '_gerekce')[:70]}")

ölçüt          v1  v2   v2 gerekçesi
dogruluk        4   4   Tarih (3 Eylül 2026) ve ikinci tahsilatın iadeye önerilmesi kaynak ve 
eksiksizlik     5   4   İki kez kesildiği doğrulanmış ve iade talebinin onaya iletildiği söyle
banka_dili      3   4   "Siz" dili kullanılmış, kesin vaat yok ve "onaya iletildi" ifadesiyle 
empati          2   5   Rehberdeki "Bu durumun can sıkıcı olduğunu anlıyoruz" kalıbıyla tek cü


**Tam test setinde ne çıktı?** Tek talep bir A/B testi değildir; karar 37 satırlık test setinde verilir.
Eğitmenin koşusunda (taslak v1 ve v2, 2'şer tekrar) sonuçlar şöyleydi:

| Metrik (ortalama, 2 koşu) | Bulut v1 | Bulut v2 | Yerel v1 | Yerel v2 |
|---|---|---|---|---|
| Davranış doğruluğu | %93 | **%97** | %58 | **%70** |
| Olması gereken bilgi | %90 | **%98** | %55 | **%67** |
| Kişisel veri (modele / yanıtta) | 0 / 0 | 0 / 0 | 0 / 0 | 0 / 0 |
| Ortalama süre | 15,9 sn | 13,2 sn | 22,8 sn | 19,3 sn |
| Hakem: banka dili / empati (1-5) | 3,3 / 3,1 | **3,9 / 4,0** | – | – |

Bulutta v2 kararda biraz önde (+4); asıl farkı **içerik** ve **hakem** gösteriyor. Yerelde v2 küçük modele açıkça yardım ediyor.
Bu yüzden v2 varsayılan sürüm oldu.

Görev 2'de (Lab 2) bu ölçümü siz yapacaksınız: `python eval/ab_testi.py --a v1 --b v2 --tekrar 2`

---

## 💬 Tartışma

1. Bölüm 1'de zayıf istem de "doğru" bir şey söyledi. O halde dört bölümlü şablonun asıl kazancı nedir: doğruluk mu, tutarlılık mı, denetlenebilirlik mi?
2. Bölüm 2'de modele `[KART_1]` gitti. Model kart numarasını hiç görmeden müşteriye "0007 ile biten kartınız" demeyi nasıl başardı? Eşleme tablosu kaybolsaydı ne olurdu?
3. Self-check de bir model. Kendi hatasını göremeyen bir model, kendini kontrol ederken neyi yakalar, neyi kaçırır? Bu yüzden hangi kontrolleri **kod** yapıyor?
4. A/B testinde v2 bir metrikte iyileşip başka birinde kötüleşirse hangisine bakarak karar verirsiniz? Hakem puanı ile test seti metriği çelişirse hangisine güvenirsiniz?